# 🎙️ Behavioral Signals Deepfakes API Demo

This notebook demonstrates how to use the Behavioral Signals Python SDK to detect deepfake audio using both **batch** and **streaming** modes.

## 🚀 How to Use This Notebook

1. **Run the first cell** to install required packages (this takes ~30 seconds)
2. **Enter your API credentials** in the second code cell
3. **Run all remaining cells** by clicking `Runtime > Run all` or pressing `Ctrl+F9`
4. **Watch the demos execute** - results will appear automatically!

💡 **Tip**: You can run cells individually by clicking the play button ▶️ or pressing `Shift+Enter`

## What You'll Learn
1. How to use the **Batch API** for processing complete audio files
2. How to use the **Streaming API** for real-time audio analysis
3. How to interpret deepfake detection results

## Prerequisites
- Behavioral Signals API credentials (CID and API_KEY) - [Get credentials](https://behavioralsignals.com)
- No local dependencies required - everything runs in Colab!

---
## 📦 Setup

First, let's install the required packages.

In [ ]:
# Install packages with compatible protobuf version for Colab
!pip install -q 'protobuf<6.0.0' behavioralsignals datasets soundfile

### 🔑 Enter Your API Credentials

Replace the placeholder values with your actual credentials from Behavioral Signals.

In [ ]:
# Enter your credentials here
CID = "your_cid_here"
API_KEY = "your_api_key_here"

# Verify credentials are set
if CID == "your_cid_here" or API_KEY == "your_api_key_here":
    print("⚠️ Please update CID and API_KEY with your actual credentials!")
else:
    print("✅ Credentials set successfully!")

### 📊 Load Sample Dataset

We'll use a sample dataset from HuggingFace that contains both genuine and deepfake audio samples.

In [ ]:
from datasets import load_dataset
import soundfile as sf
import os

# Load the deepfake detection demo dataset
print("Loading dataset from HuggingFace...")
dataset = load_dataset("behavioralsignals/deepfake-detection-demo", split="test")

print(f"✅ Loaded {len(dataset)} audio samples")
print(f"\nDataset columns: {dataset.column_names}")
print(f"\nFirst few samples:")
for i in range(min(3, len(dataset))):
    print(f"  - {dataset[i]['language']}: {dataset[i]['label']}")

---
## 🎯 Demo 1: Batch API

The **Batch API** is ideal for processing complete audio files. You submit an audio file and poll for results.

### How it works:
1. Upload an audio file → Get a Process ID (PID)
2. Poll the API to check processing status
3. Retrieve results when complete

In [ ]:
from behavioralsignals import Client
import time
import pandas as pd

# Initialize the client
client = Client(cid=CID, api_key=API_KEY)
deepfakes_client = client.deepfakes

# Select a sample audio (let's use the first English sample)
sample_idx = 0  # English bonafide sample
sample = dataset[sample_idx]

print(f"📁 Processing sample: {sample['language']} - {sample['label']}")
print(f"   Duration: {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f}s")

# Save audio to temporary file
temp_audio_path = "/tmp/sample_audio.wav"
sf.write(temp_audio_path, sample['audio']['array'], sample['audio']['sampling_rate'])

# Step 1: Upload the audio file
print("\n🚀 Uploading audio file...")
upload_response = deepfakes_client.upload_audio(file_path=temp_audio_path)
pid = upload_response.pid
print(f"✅ Upload successful! Process ID: {pid}")

# Step 2: Poll for results
print("\n⏳ Waiting for processing to complete...")
last_status = None
while True:
    process = deepfakes_client.get_process(pid=pid)
    
    if process.is_completed:
        if last_status != process.statusmsg:
            print("✅ Processing complete!")
        break
    elif process.is_processing:
        if last_status != process.statusmsg:
            print("   Processing audio...")
    elif process.is_pending:
        if last_status != process.statusmsg:
            print("   API is busy, waiting...")
    
    last_status = process.statusmsg
    time.sleep(1.0)

# Step 3: Get the results
result = deepfakes_client.get_result(pid=pid)

print("\n" + "="*60)
print("📊 BATCH API RESULTS")
print("="*60)
print(f"\n🎯 Prediction: {result.prediction.upper()}")
print(f"📈 Confidence Score: {result.score:.4f}")
print(f"✓ Actual Label: {sample['label']}")

if result.prediction.lower() == sample['label'].lower():
    print("\n✅ Correct prediction!")
else:
    print("\n❌ Incorrect prediction")

# Display results in a nice table
results_df = pd.DataFrame([{
    'Language': sample['language'],
    'Actual Label': sample['label'],
    'Predicted': result.prediction,
    'Confidence Score': f"{result.score:.4f}",
    'Status': '✅ Correct' if result.prediction.lower() == sample['label'].lower() else '❌ Wrong'
}])

print("\n" + results_df.to_string(index=False))

---
## 🌊 Demo 2: Streaming API

The **Streaming API** is ideal for real-time audio analysis. Audio is sent in chunks and you receive responses as processing happens.

### How it works:
1. Create an audio stream (iterator of audio chunks)
2. Stream audio chunks to the API
3. Receive real-time responses for each segment

In [ ]:
from behavioralsignals import StreamingOptions
from behavioralsignals.utils import make_audio_stream

# Let's use a different sample for streaming (first spoofed sample)
spoofed_samples = [i for i, item in enumerate(dataset) if item['label'].lower() == 'spoof']
if spoofed_samples:
    sample_idx = spoofed_samples[0]
    sample = dataset[sample_idx]
else:
    sample_idx = 1
    sample = dataset[sample_idx]

print(f"📁 Streaming sample: {sample['language']} - {sample['label']}")
print(f"   Duration: {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f}s")

# Save audio to temporary file
temp_audio_path = "/tmp/streaming_sample.wav"
sf.write(temp_audio_path, sample['audio']['array'], sample['audio']['sampling_rate'])

# Create audio stream (chunks of 0.25 seconds)
print("\n🌊 Creating audio stream...")
audio_stream, sample_rate = make_audio_stream(temp_audio_path, chunk_size=0.25)

# Configure streaming options
options = StreamingOptions(
    sample_rate=sample_rate,
    encoding="LINEAR_PCM",
    level="all"  # Get both segment and utterance level results
)

# Stream audio and collect responses
print("🚀 Streaming audio to API...\n")
responses = deepfakes_client.stream_audio(audio_stream=audio_stream, options=options)

segment_results = []
utterance_result = None

for i, response in enumerate(responses):
    if hasattr(response, 'segment_result') and response.segment_result:
        seg = response.segment_result
        segment_results.append({
            'Segment': f"#{i+1}",
            'Prediction': seg.prediction,
            'Score': f"{seg.score:.4f}"
        })
        print(f"📍 Segment #{i+1}: {seg.prediction} (score: {seg.score:.4f})")
    
    if hasattr(response, 'utterance_result') and response.utterance_result:
        utterance_result = response.utterance_result

print("\n" + "="*60)
print("📊 STREAMING API RESULTS")
print("="*60)

if utterance_result:
    print(f"\n🎯 Final Prediction: {utterance_result.prediction.upper()}")
    print(f"📈 Confidence Score: {utterance_result.score:.4f}")
    print(f"✓ Actual Label: {sample['label']}")
    
    if utterance_result.prediction.lower() == sample['label'].lower():
        print("\n✅ Correct prediction!")
    else:
        print("\n❌ Incorrect prediction")

# Display segment results in a table
if segment_results:
    print("\n📋 Segment-by-Segment Results:")
    segments_df = pd.DataFrame(segment_results)
    print(segments_df.to_string(index=False))

---
## 🎯 Batch Processing Multiple Samples

Let's process multiple samples to see how the API performs across different languages and types.

In [ ]:
# Process first 5 samples using batch API
batch_results = []

print("Processing multiple samples...\n")
for idx in range(min(5, len(dataset))):
    sample = dataset[idx]
    
    # Save to temp file
    temp_path = f"/tmp/batch_sample_{idx}.wav"
    sf.write(temp_path, sample['audio']['array'], sample['audio']['sampling_rate'])
    
    # Upload and process
    print(f"[{idx+1}/5] Processing {sample['language']} - {sample['label']}...")
    upload_response = deepfakes_client.upload_audio(file_path=temp_path)
    pid = upload_response.pid
    
    # Poll for completion
    while True:
        process = deepfakes_client.get_process(pid=pid)
        if process.is_completed:
            break
        time.sleep(0.5)
    
    # Get results
    result = deepfakes_client.get_result(pid=pid)
    
    batch_results.append({
        'Language': sample['language'],
        'Actual': sample['label'],
        'Predicted': result.prediction,
        'Score': result.score,
        'Correct': '✅' if result.prediction.lower() == sample['label'].lower() else '❌'
    })
    
    print(f"    → {result.prediction} (score: {result.score:.4f}) {batch_results[-1]['Correct']}")

# Display summary
print("\n" + "="*60)
print("📊 BATCH PROCESSING SUMMARY")
print("="*60 + "\n")

results_df = pd.DataFrame(batch_results)
results_df['Score'] = results_df['Score'].apply(lambda x: f"{x:.4f}")
print(results_df.to_string(index=False))

# Calculate accuracy
correct_count = sum(1 for r in batch_results if r['Correct'] == '✅')
accuracy = (correct_count / len(batch_results)) * 100
print(f"\n🎯 Accuracy: {correct_count}/{len(batch_results)} ({accuracy:.1f}%)")

---
## 🎉 Summary

You've successfully learned how to:

✅ **Use the Batch API** for complete audio file processing
- Upload audio files
- Poll for processing status
- Retrieve and interpret results

✅ **Use the Streaming API** for real-time analysis
- Create audio streams
- Process audio in chunks
- Receive segment-level and utterance-level results

✅ **Batch process multiple samples** for evaluation

### Next Steps
- Try with your own audio files
- Experiment with different response levels (`segment`, `utterance`, `all`)
- Integrate the SDK into your applications

### Resources
- 📚 [SDK Documentation](https://github.com/BehavioralSignalTechnologies/behavioralsignals-python)
- 🎯 [HuggingFace Dataset](https://huggingface.co/datasets/behavioralsignals/deepfake-detection-demo)
- 🌐 [Behavioral Signals Website](https://behavioralsignals.com)